# **Тема:** RAG (Retrieval-Augmented Generation) с фреймворком LangChain


Разработка RAG-пайплайна


**Задачи:**
* Загрузить набор текстовых документов (например, статей из датасета arXiv Dataset: https://www.kaggle.com/datasets/Cornell-University/arxiv)
* Разбить текст на чанки с помощью Langchain text splitter
* Создать векторный индекс с помощью FAISS и sentence-transformers
* Реализовать langchain-цепочку, которая производим семантический поиск и формирует промпт для LLM (локальной или через Groq/OpenRouter)
* Протестировать систему на нескольких вопросах, оценить качество ответов


**Библиотеки:** langchain, huggingface, faiss-cpu, sentence-transformers

**Ожидаемый результат:** Colab-ноутбук с рабочим прототипом наукоёмкой (например, разработанной на основе текстов ArXiv) RAG-системы, примерами её ответов и качественным анализом, представленным в текстовых блоках


**Участники проекта:** Игорь Васильев, Глафира Ломакина

## Загрузить набор текстовых документов

### Датасет с метаданными к статьям

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# путь к файлу датасета
file_path = "arxiv-metadata-oai-snapshot.json"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "Cornell-University/arxiv",
  file_path,
  pandas_kwargs={"lines": True, "nrows": 10000}, # указываем число строк
)
df["id"] = df["id"].apply(lambda x: str(x).zfill(9))

/tmp/ipykernel_13913/3709853837.py:7: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'arxiv' dataset.


In [ ]:
# заглянем
df.head(2)

,id,submitter,authors,title,comments,journal-ref,doi,report-no,categories,license,abstract,versions,update_date,authors_parsed
0,0704.0001,Pavel Nadolsky,"C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...",Calculation of prompt diphoton production cros...,"37 pages, 15 figures; published version","Phys.Rev.D76:013009,2007",10.1103/PhysRevD.76.013009,ANL-HEP-PR-07-12,hep-ph,None,A fully differential calculation in perturba...,"[{'version': 'v1', 'created': 'Mon, 2 Apr 2007...",2008-11-26,"[[Balázs, C., ], [Berger, E. L., ], [Nadolsky,..."
1,0704.0002,Louis Theran,Ileana Streinu and Louis Theran,Sparsity-certifying Graph Decompositions,To appear in Graphs and Combinatorics,None,None,None,math.CO cs.CG,http://arxiv.org/licenses/nonexclusive-distrib...,"We describe a new algorithm, the $(k,\ell)$-...","[{'version': 'v1', 'created': 'Sat, 31 Mar 200...",2008-12-13,"[[Streinu, Ileana, ], [Theran, Louis, ]]"


### Подбор статей

внимание на столбец id: по нему можно добраться до самой статьи

 https://arxiv.org/abs/{id}: посмотреть страницу статьи с абстрактом

 https://arxiv.org/pdf/{id}: скачать

отбираем статьи по Quantitative Biology:

In [ ]:
# отфильтруем df по заданному признаку и получим список id для дальнейшего сохранения
ids = df[df['categories'].str.contains('q-bio', na=False, regex=True)]['id'].tolist()
print(len(ids))

173


вошло: **173/10000**

### Загрузка

In [ ]:
!pip install langchain_community langchain_text_splitters pypdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# загружаем с PyPDFLoader, используя https://arxiv.org/pdf/{id}
documents = []
for id in ids:
    try:
        url = f"https://arxiv.org/pdf/{id}"
        loader = PyPDFLoader(url, mode="single")
        docs = loader.load()
        documents.extend(docs)
    except Exception as e:
        print(f"не удалось загрузить {id}. {e}")
# получаем список Document объектов
print(f"загружено {len(documents)} документов")

ERROR:pypdf._cmap:Advanced encoding [] not implemented yet


не удалось загрузить 00704.139. Check the url of your file; returned status code 404
не удалось загрузить 000704.22. Check the url of your file; returned status code 404
не удалось загрузить 00704.226. Check the url of your file; returned status code 404
не удалось загрузить 00704.364. Check the url of your file; returned status code 404
не удалось загрузить 00704.373. Check the url of your file; returned status code 404
не удалось загрузить 00705.103. Check the url of your file; returned status code 404
не удалось загрузить 00705.146. Check the url of your file; returned status code 404
не удалось загрузить 00705.149. Check the url of your file; returned status code 404
не удалось загрузить 00705.271. Check the url of your file; returned status code 404


не удалось загрузить 00705.366. Check the url of your file; returned status code 404
не удалось загрузить 00705.369. Check the url of your file; returned status code 404


не удалось загрузить 00705.463. Check the url of your file; returned status code 404
не удалось загрузить 00706.076. Check the url of your file; returned status code 404


загружено 160 документов


## Разбить на чанки

In [ ]:
# выбрать тип text splitter'а под нашу задачу
#чистка
import re
def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\n\s*\d+\s*\n', ' ', text)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\S+@\S+\.\S+', '', text)
    text = ' '.join(text.split())
    return text.strip()
for doc in documents:
    doc.page_content = clean_text(doc.page_content)
print(documents[0])
# например, по статье: https://www.geeksforgeeks.org/artificial-intelligence/text-splitter-in-langchain/
# сплиттер -- RecursiveCharacterTextSplitter: работает с длинными текстами и делит чанки с учётом семантики содержимого
# использовать split_documents
# использовать chunk_size, chunk_overlap для настройки
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)
# сохранить в chunks
chunks = text_splitter.split_documents(documents)
print("пример чанка:")
print(chunks[0].page_content[:800])

page_content='arXiv:0704.0021v2 [nlin.PS] 24 Jul 2007 Molecular Synchronization Waves in Arrays of Allosterical ly Regulated Enzymes Vanessa Casagrande, 1 Yuichi Togashi,2, ∗ and Alexander S. Mikhailov 2, † 1Hahn-Meitner-Institut, Glienicker Straße 100, 14109 Berl in, Germany 2Fritz-Haber-Institut der Max-Planck-Gesellschaft, Fara dayweg 4-6, 14195 Berlin, Germany Spatiotemporal pattern formation in a product-activated e nzymic reaction at high enzyme con- centrations is investigated. Stochastic simulations show that catalytic turnover cycles of individual enzymes can become coherent and that complex wave patterns o f molecular synchronization can develop. The analysis based on the mean-ﬁeld approximation indicates that the observed patterns result from the presence of Hopf and wave bifurcations in the considered system. PACS numbers: 82.40.Ck, 87.18.Pj, 82.39.Fk, 05.45.Xt Molecular machines, such as molecular motors, ion pumps and some enzymes, play a fundamental role in biological ce

overlap 25% для лучшего сохранения контекста

In [ ]:
print(len(chunks))

11364


## Создать векторный индекс с помощью faiss-cpu и sentence-transformers

In [ ]:
!pip install faiss-cpu sentence-transformers langchain-huggingface -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.2 MB/s eta 0:00:00


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

модель: all-MiniLM-L6-v2, подойдёт для работы с небольшими чанками вроде наших

In [ ]:
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)
print(f"Обработано чанков: {vectorstore.index.ntotal}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Обработано чанков: 11364


## Реализовать цепочку

In [ ]:
!pip install langchain_core langchain_classic langchain_openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 22.0 MB/s eta 0:00:00


In [ ]:
import json
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from google.colab import userdata

In [ ]:
from langchain_openai import ChatOpenAI

In [ ]:
# задаем параметры
GEN_MODEL_ID = "arcee-ai/trinity-large-preview:free" # должна справиться с англ текстом
TOP_K = 3 # чтобы не засорять контекст модели нерелевантными текстами
QUESTION = "What are stem cells?"
PROMPT = PromptTemplate(
    input_variables=["context", "input"],
    template="""You are an expert in biology. Answer the following question(s):
    Question: {input}
    Context: {context}
    Answer:
    only provide information if there is enough context. in any other case state you could not find exact sources on the matter and proceed to find anything similar to the query"""
)
TASK = "text-generation"

hugging face плохо работала. подгрузили через openrouter

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
OPEN_ROUTER_API_KEY = userdata.get('OPEN')
llm = ChatOpenAI(
    api_key=OPEN_ROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    model=GEN_MODEL_ID,
)

In [ ]:
def clip_text(text, threshold=100):
    return f"{text[:threshold]}..." if len(text) > threshold else text

In [ ]:
question_answer_chain = create_stuff_documents_chain(llm, PROMPT)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
def vopros(question=QUESTION):
    resp_dict = rag_chain.invoke({"input": question})
    clipped_answer = clip_text(resp_dict["answer"], threshold=350)
    print(f"Question:\n{resp_dict['input']}\n\nAnswer:\n{clipped_answer}")
    for i, doc in enumerate(resp_dict["context"]):
        print()
        print(f"Source {i + 1}:")
        print(f"  text: {json.dumps(clip_text(doc.page_content, threshold=350))}")
        for key in doc.metadata:
            if key != "pk":
                val = doc.metadata.get(key)
                clipped_val = clip_text(val) if isinstance(val, str) else val
                print(f"  {key}: {clipped_val}")
vopros()

Question:
What are stem cells?

Answer:
Stem cells are undifferentiated cells that have the unique ability to develop into various specialized cell types in the body. They serve as a repair system, capable of dividing and renewing themselves over long periods, and can differentiate into cells with specific functions, such as muscle cells, nerve cells, or blood cells. There are two main t...

Source 1:
  text: "Gage F. (2002) Stem cells for a new clinical neuroscience- Introduction. Clinical neuroscience research 2 (1-2) 31. Cage F.,(2007) 3rd Stem Cell Research & Therapeutics conference, March 22-23, San Diego, CA 32. N. Mekel-Bobrov, S. L. Gilbert, P. D. Evans, E. J. Vallender, J. R. Anderson, R. R. Hudson, S. A. Tishkoff, B. T. Lahn, Ongoing Adaptive E..."
  producer: Acrobat Distiller 6.0 (Windows)
  creator: Acrobat PDFMaker 6.0 for Word
  creationdate: 2007-08-02T11:25:55-04:00
  moddate: 2007-08-02T11:26:55-04:00
  title: Mortality study establishes universal law of express adapt

## Протестировать на нескольких примерах, оченить качество

In [ ]:
vopros("What are stem cells?") # фактический
print('\n\n')
vopros("What is the importance of stem cells research for modern medicine?") # обобщающий
print('\n\n')
vopros("What is the difference between regular cells and stem cells?") # уточняющий
print('\n\n')

Question:
What are stem cells?

Answer:
Stem cells are unique cells with the remarkable ability to develop into many different cell types in the body during early life and growth. They serve as a sort of internal repair system, dividing essentially without limit to replenish other cells as long as the person or animal is still alive. When a stem cell divides, each new cell has the potent...

Source 1:
  text: "Gage F. (2002) Stem cells for a new clinical neuroscience- Introduction. Clinical neuroscience research 2 (1-2) 31. Cage F.,(2007) 3rd Stem Cell Research & Therapeutics conference, March 22-23, San Diego, CA 32. N. Mekel-Bobrov, S. L. Gilbert, P. D. Evans, E. J. Vallender, J. R. Anderson, R. R. Hudson, S. A. Tishkoff, B. T. Lahn, Ongoing Adaptive E..."
  producer: Acrobat Distiller 6.0 (Windows)
  creator: Acrobat PDFMaker 6.0 for Word
  creationdate: 2007-08-02T11:25:55-04:00
  moddate: 2007-08-02T11:26:55-04:00
  title: Mortality study establishes universal law of express adapt

как видно, модель адекватно воспринимает промпт и не начинает галлюцинировать в случае, если непосредственный ответ на вопрос найти не удаётся, а ищет схожие чанки. возможно, 160 статей мало для вопроса непосредственно об особенностях стволовых клеток, но определение в них явно встречается, что модель и считала.

на третий вопрос, например, модель ответила чанками из одной и той же статьи, касающейся клеточного устройства фасеток глаза у членистоногих. но при том в качестве третьего источника модель извлекла чанк с введением к статье о стволовых клетках и значимости для клинических исследований нейронов: возможно, была допущена ошибка при дроблении на чанки, хотя число их совпало с числом статей, была настройка mode=single



---



# Критерии оценки

Работа проверяется по следующим критериям (максимум 10 баллов):

### Загрузка и подготовка данных (2 балла)
- [ ] 0.5 балла: выбран критерий подбора материалов
- [ ] 0.5 балла: загружено не менее 100 записей/статей
- [ ] 0.5 балла: тексты успешно извлечены из источника
- [ ] 0.5 балла: данные приведены к формату, пригодному для чанкинга (очистка, объединение полей)

### Чанкинг (2 балла)
- [ ] 0.5 балла: выбран подходящий тип сплиттера (RecursiveCharacterTextSplitter, HTMLHeaderTextSplitter и т.д.)
- [ ] 0.5 балла: обоснован выбор размера чанка и перекрытия (например, "512 токенов, overlap 20% для сохранения контекста")
- [ ] 0.5 балла: чанки созданы и не содержат явных артефактов (оборванных слов)
- [ ] 0.5 балла: количество чанков соответствует ожидаемому (не 1 и не 100500 на документ)

### Векторное хранилище (1 балл)
- [ ] 0.5 балла: выбрана адекватная эмбеддинг-модель (например, all-MiniLM-L6-v2 для русского/английского)
- [ ] 0.5 балла: индекс создан


### Реализация цепочки (3 балла)
- [ ] 0.5 балла: выбрана LLM
- [ ] 1 балл: промпт, QUESTION, TASK составлены корректно
- [ ] 0.5 балла: обоснован заданный TOP_K
- [ ] 0.5 балла: ответ генерируется на основе найденных чанков (видно по содержанию)
- [ ] 0.5 балла: обработан случай отсутствия информации в контексте

### Тестирование и анализ (2 балла)
- [ ] 0.5 балла: задано минимум 3 разнотипных вопроса (фактический, обобщающий, уточняющий)
- [ ] 0.5 балла: для каждого вопроса показан и проанализирован ответ
- [ ] 0.5 балла: в анализе указано, какие чанки использовались и почему
- [ ] 0.5 балла: сделан вывод о качестве работы системы (что получилось, что нет, гипотезы почему)
